<a href="https://colab.research.google.com/github/Aymn74/Fine_Tunung_gemma_3_1b_On_MedQA/blob/main/fine_tunung_gemma_3_1b_on_medqa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
%%capture
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it",
    max_seq_length = 1024,
    load_in_4bit = True,
)

print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
from datasets import load_dataset

dataset = load_dataset("GBaker/MedQA-USMLE-4-options")

print(dataset)
print(dataset["train"][0])

README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

phrases_no_exclude_train.jsonl:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

phrases_no_exclude_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 10178
    })
    test: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1273
    })
})
{'question': 'A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?', 'answer': 'Nitrofurantoin', 'options': {'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C':

In [ ]:
def format_medqa(example):
    options_text = "\n".join([f"{key}. {value}" for key, value in example["options"].items()])

    prompt = f"""Answer the following medical multiple-choice question.

Question:
{example["question"]}

Options:
{options_text}

Return the correct option letter and answer only.
"""

    response = f"{example['answer_idx']}. {example['answer']}"

    return {
        "text": f"<bos><start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n{response}<end_of_turn>"
    }

train_dataset = dataset["train"].select(range(500)).map(format_medqa)

print(train_dataset[0]["text"])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

<bos><start_of_turn>user
Answer the following medical multiple-choice question.

Question:
A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?

Options:
A. Ampicillin
B. Ceftriaxone
C. Doxycycline
D. Nitrofurantoin

Return the correct option letter and answer only.
<end_of_turn>
<start_of_turn>model
D. Nitrofurantoin<end_of_turn>


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        seed = 3407,
    ),
)

trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 6,522,880 of 1,006,408,832 (0.65% trained)


Step,Training Loss
1,3.849796
2,3.812227
3,3.751406
4,3.472387
5,3.249010
6,2.694052
7,3.052804
8,2.832150
9,2.409170
10,2.559779


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-30.


TrainOutput(global_step=30, training_loss=2.4164175271987913, metrics={'train_runtime': 322.2245, 'train_samples_per_second': 0.745, 'train_steps_per_second': 0.093, 'total_flos': 305653015153152.0, 'train_loss': 2.4164175271987913, 'epoch': 0.48})

In [ ]:
FastLanguageModel.for_inference(model)

prompt = """<bos><start_of_turn>user
Answer the following medical multiple-choice question.

Question:
A pregnant woman has dysuria without fever or flank pain. What is the best treatment?

Options:
A. Doxycycline
B. Nitrofurantoin
C. Vancomycin
D. Oseltamivir

Return the correct option letter and answer only.
<end_of_turn>
<start_of_turn>model
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=30,
    temperature=0.1,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

Both `max_new_tokens` (=30) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<bos><bos><start_of_turn>user
Answer the following medical multiple-choice question.

Question:
A pregnant woman has dysuria without fever or flank pain. What is the best treatment?

Options:
A. Doxycycline
B. Nitrofurantoin
C. Vancomycin
D. Oseltamivir

Return the correct option letter and answer only.
<end_of_turn>
<start_of_turn>model
B. Nitrofurantoin<end_of_turn>


In [ ]:
model.save_pretrained("medqa_gemma_lora")
tokenizer.save_pretrained("medqa_gemma_lora")

Unsloth: Restored added_tokens_decoder metadata in medqa_gemma_lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in medqa_gemma_lora.


('medqa_gemma_lora/tokenizer_config.json',
 'medqa_gemma_lora/chat_template.jinja',
 'medqa_gemma_lora/tokenizer.json')

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "medqa_gemma_lora",
    max_seq_length = 1024,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.5.2: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [ ]:
from huggingface_hub import login

login()

In [ ]:
repo_id = "Ay74/medqa-gemma-1b-lora"

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

README.md:   0%|          | 0.00/574 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|2         |  559kB / 26.1MB            

Saved model to https://huggingface.co/Ay74/medqa-gemma-1b-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpsq93u47b/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /tmp/tmpsq93u47b.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpsq93u47b/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

  ...psq93u47b/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            